# Step 3 – nnU-Net v2 Planning and Preprocessing

This notebook runs the nnU-Net v2 planning and preprocessing stage
for Dataset101_BraTS2020.

Input:
- nnUNet_raw/Dataset101_BraTS2020

Output:
- nnUNet_preprocessed/Dataset101_BraTS2020

Notes:
- Preprocessed data is stored on local Colab disk for performance.
- Optionally, the preprocessed folder can be archived after completion.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)

2.9.0+cu126
True
12.6


In [ ]:
# No need to pip install -e ., as now we do not need to change any element
!pip install -U nnunetv2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.9/212.9 kB 6.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 10.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 8.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)

2.10.0+cu128
True
12.8


## 1. Set nnU-Net environment variables

In [ ]:
import os

# Path to nnU-Net raw data (can be on Google Drive)
os.environ["nnUNet_raw"] = "/content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_raw"

# Path to nnU-Net preprocessed data (must be on local disk for speed)
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"

# Path to nnU-Net results (can be on Google Drive)
os.environ["nnUNet_results"] = "/content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results"

print("nnUNet_raw:", os.environ["nnUNet_raw"])
print("nnUNet_preprocessed:", os.environ["nnUNet_preprocessed"])
print("nnUNet_results:", os.environ["nnUNet_results"])

nnUNet_raw: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_raw
nnUNet_preprocessed: /content/nnUNet_preprocessed
nnUNet_results: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results


## 2. Sanity check: dataset presence

In [ ]:
from pathlib import Path

dataset_dir = Path(os.environ["nnUNet_raw"]) / "Dataset101_BraTS2020"

assert dataset_dir.exists()
assert (dataset_dir / "dataset.json").exists()
assert (dataset_dir / "imagesTr").exists()
assert (dataset_dir / "labelsTr").exists()

print("Dataset structure OK.")

Dataset structure OK.


## 3. Run nnU-Net v2 planning and preprocessing

### 3.1 Default Planner (PlainConvUNet) => nnUNetPlans.json

In [ ]:
!nnUNetv2_plan_and_preprocess -d 101 # --verify_dataset_integrity

Fingerprint extraction...
Dataset101_BraTS2020
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer
100% 369/369 [02:08<00:00,  2.86it/s]
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [139. 170. 138.], 3d_lowres: [139, 170, 138]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 105, 'patch_size': (np.int64(192), np.int64(160)), 'median_image_size_in_voxels': array([170., 138.]), 'spacing': array([1., 1.]), 'normalization_schemes': ['ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization'], 'use_mask

### 3.2 Planner ResidualEncoderUNet XL => nnUNetResEncUNetXLPlans.json

In [ ]:
!nnUNetv2_plan_experiment -d 101 -pl nnUNetPlannerResEncXL -overwrite_plans_name nnUNetResEncUNetXLPlans

Fingerprint extraction...
Dataset101_BraTS2020
Experiment planning...
Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [139. 170. 138.], 3d_lowres: [139, 170, 138]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 301, 'patch_size': (np.int64(192), np.int64(160)), 'median_image_size_in_voxels': array([170., 138.]), 'spacing': array([1., 1.]), 'normalization_schemes': ['ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization'], 'use_mask_for_norm': [True, True, True, True], 'resampling_fn_data': 'resample_data_or_seg_to_shape', 'resampling_fn_seg': 'resample_data_or_seg_to_shape', 'resampling_fn_data_kwargs': {'is_seg': False, 'order': 3, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_seg_kwargs': {'is_seg': True, 'order': 1, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_probabilities': 'resample_data_or_seg_to_shape

```
Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [139. 170. 138.], 3d_lowres: [139, 170, 138]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 444, 'patch_size': (np.int64(192), np.int64(160)), 'median_image_size_in_voxels': array([170., 138.]), 'spacing': array([1., 1.]), 'normalization_schemes': ['ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization'], 'use_mask_for_norm': [True, True, True, True], 'resampling_fn_data': 'resample_data_or_seg_to_shape', 'resampling_fn_seg': 'resample_data_or_seg_to_shape', 'resampling_fn_data_kwargs': {'is_seg': False, 'order': 3, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_seg_kwargs': {'is_seg': True, 'order': 1, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_probabilities': 'resample_data_or_seg_to_shape', 'resampling_fn_probabilities_kwargs': {'is_seg': False, 'order': 1, 'order_z': 0, 'force_separate_z': None}, 'architecture': {'network_class_name': 'dynamic_network_architectures.architectures.unet.ResidualEncoderUNet', 'arch_kwargs': {'n_stages': 6, 'features_per_stage': (32, 64, 128, 256, 512, 512), 'conv_op': 'torch.nn.modules.conv.Conv2d', 'kernel_sizes': ((3, 3), (3, 3), (3, 3), (3, 3), (3, 3), (3, 3)), 'strides': ((1, 1), (2, 2), (2, 2), (2, 2), (2, 2), (2, 2)), 'n_blocks_per_stage': (1, 3, 4, 6, 6, 6), 'n_conv_per_stage_decoder': (1, 1, 1, 1, 1), 'conv_bias': True, 'norm_op': 'torch.nn.modules.instancenorm.InstanceNorm2d', 'norm_op_kwargs': {'eps': 1e-05, 'affine': True}, 'dropout_op': None, 'dropout_op_kwargs': None, 'nonlin': 'torch.nn.LeakyReLU', 'nonlin_kwargs': {'inplace': True}}, '_kw_requires_import': ('conv_op', 'norm_op', 'dropout_op', 'nonlin')}, 'batch_dice': True}

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer
3D fullres U-Net configuration:
{'data_identifier': 'nnUNetPlans_3d_fullres', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 5, 'patch_size': (np.int64(160), np.int64(192), np.int64(160)), 'median_image_size_in_voxels': array([139., 170., 138.]), 'spacing': array([1., 1., 1.]), 'normalization_schemes': ['ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization'], 'use_mask_for_norm': [True, True, True, True], 'resampling_fn_data': 'resample_data_or_seg_to_shape', 'resampling_fn_seg': 'resample_data_or_seg_to_shape', 'resampling_fn_data_kwargs': {'is_seg': False, 'order': 3, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_seg_kwargs': {'is_seg': True, 'order': 1, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_probabilities': 'resample_data_or_seg_to_shape', 'resampling_fn_probabilities_kwargs': {'is_seg': False, 'order': 1, 'order_z': 0, 'force_separate_z': None}, 'architecture': {'network_class_name': 'dynamic_network_architectures.architectures.unet.ResidualEncoderUNet', 'arch_kwargs': {'n_stages': 6, 'features_per_stage': (32, 64, 128, 256, 320, 320), 'conv_op': 'torch.nn.modules.conv.Conv3d', 'kernel_sizes': ((3, 3, 3), (3, 3, 3), (3, 3, 3), (3, 3, 3), (3, 3, 3), (3, 3, 3)), 'strides': ((1, 1, 1), (2, 2, 2), (2, 2, 2), (2, 2, 2), (2, 2, 2), (2, 2, 2)), 'n_blocks_per_stage': (1, 3, 4, 6, 6, 6), 'n_conv_per_stage_decoder': (1, 1, 1, 1, 1), 'conv_bias': True, 'norm_op': 'torch.nn.modules.instancenorm.InstanceNorm3d', 'norm_op_kwargs': {'eps': 1e-05, 'affine': True}, 'dropout_op': None, 'dropout_op_kwargs': None, 'nonlin': 'torch.nn.LeakyReLU', 'nonlin_kwargs': {'inplace': True}}, '_kw_requires_import': ('conv_op', 'norm_op', 'dropout_op', 'nonlin')}, 'batch_dice': False}

Plans were saved to /content/nnUNet_preprocessed/Dataset101_BraTS2020/nnUNetResEncUNetXLPlans.json
```

In [ ]:
!ls

drive  nnUNet_preprocessed  sample_data


> we need to run training to get splits_final.json file, as it only appears when training. (do_split method) was defined in nnUnet/trainer so running trainning (just minutes to get splits_final.json is crucial)

In [ ]:
!nnUNetv2_train -h

usage: nnUNetv2_train [-h] [-tr TR] [-p P]
                      [-pretrained_weights PRETRAINED_WEIGHTS]
                      [-num_gpus NUM_GPUS] [--npz] [--c] [--val] [--val_best]
                      [--disable_checkpointing] [-device DEVICE]
                      dataset_name_or_id configuration fold

positional arguments:
  dataset_name_or_id    Dataset name or ID to train with
  configuration         Configuration that should be trained
  fold                  Fold of the 5-fold cross-validation. Should be an int
                        between 0 and 4.

options:
  -h, --help            show this help message and exit
  -tr TR                [OPTIONAL] Use this flag to specify a custom trainer.
                        Default: nnUNetTrainer
  -p P                  [OPTIONAL] Use this flag to specify a custom plans
                        identifier. Default: nnUNetPlans
  -pretrained_weights PRETRAINED_WEIGHTS
                        [OPTIONAL] path to nnU-Net checkpoint file 

In [ ]:
!nnUNetv2_train 101 3d_fullres 0


############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-02-08 14:15:14.399557: Using torch.compile...
2026-02-08 14:15:16.208400: do_dummy_2d_data_aug: False
2026-02-08 14:15:16.212844: Creating new 5-fold cross-validation split...
2026-02-08 14:15:16.218554: Desired fold for training: 0
2026-02-08 14:15:16.221302: This split has 295 training an

In [ ]:
from pathlib import Path
import zipfile

# Dataset name
DATASET_NAME = "Dataset101_BraTS2020"

# Source dataset folder (on Colab disk)
SRC_DATASET = Path("/content/nnUNet_preprocessed") / DATASET_NAME

# Destination zip (on Google Drive)
DST_DIR = Path("/content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_preprocessed")
DST_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = DST_DIR / f"{DATASET_NAME}.zip"

assert SRC_DATASET.exists(), f"{SRC_DATASET} not found"

print(f"Zipping {DATASET_NAME} with nnUNet_preprocessed root structure...")
print("Source:", SRC_DATASET)
print("Target:", ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in SRC_DATASET.rglob("*"):
        # IMPORTANT:
        # This ensures the zip contains:
        # nnUNet_preprocessed/DatasetXXX_NAME/...
        arcname = Path("nnUNet_preprocessed") / p.relative_to(SRC_DATASET.parent)
        zf.write(p, arcname)

print("Done. Dataset-wise zip created (nnU-Net compliant).")

Zipping Dataset101_BraTS2020 with nnUNet_preprocessed root structure...
Source: /content/nnUNet_preprocessed/Dataset101_BraTS2020
Target: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_preprocessed/Dataset101_BraTS2020.zip
Done. Dataset-wise zip created (nnU-Net compliant).
